# Order dependence and cost; texture-search side experiment (`tab:app-order`)

Produces `compute_cost/{order_dependence_n6.csv, order_dependence_n20.csv, timing_n20.csv}` -- the swap-distance prose (.637) and most of `tab:app-order`'s rows. The order table's Vacher rows are unverified even against this file (see docs/INVENTORY.md section 2).

**Needs:** DTD, the CNT checkpoint, the Vacher baseline repo.

**Maintained runnable path:** none. No `experiments/*.py` script covers `tab:app-order`; run this notebook as-is.

In [ ]:
!pip install -q pot
import os, sys, torch, torch.nn.functional as F, numpy as np, glob, random, inspect, time
from pathlib import Path
from PIL import Image
import pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
DTD_ROOT = f'{BASE}/dtd/images'
OUT = f'{BASE}/corrected_rerun'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

# what does the transform actually do?
from src.inference import core as cnt_core
try:
    print(inspect.getsource(cnt_core.build_content_transform))
except Exception as e:
    import src.inference.core as c
    print("could not fetch:", e)

# ---- THE loader. Always go through load_image, via a temp file if needed. ----
_tmp_counter = [0]
def cnt_load(x, size=IMG_SIZE):
    """x: path or PIL. Always routed through CNT's load_image."""
    if isinstance(x, (str, Path)):
        return load_image(Path(x), size).to(device)
    _tmp_counter[0] += 1
    fp = f'{TMP}/t{_tmp_counter[0] % 64}.png'
    x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)

def to_pil(dec):
    a = ((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def contrast(im): return float(np.asarray(im.convert('L')).std())

def get_gen_inputs(img):
    with torch.no_grad():
        blob = model.encode(img); layout = model.splat(blob)
        return blob, layout["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     layout["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        layout = model.splat(blob_ref)
        layout["layer_level_layouts"][-1][0]["blob_features"] = bf
        layout["layer_level_layouts"][-1][0]["score_img"] = si
        return model.decode(layout)
def build_style_maps(bf, si, fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        layout=model.splat(blob_ref); si=layout["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        feat_in=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=feat_in.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    plan=u*K*v.T; plan=plan/(plan.sum(1,keepdim=True)+1e-10); return plan@tgt

def build_d(ia, ib):
    ba,bfa,sia = get_gen_inputs(ia); bb,bfb,sib = get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sm_a[L][0],sm_b[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    interp=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_ot_sym(d, eta, n_sub=4096):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sm_a[L][0],sm_b[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bfm=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        t_ab=sinkhorn_transport(af[ia],bfm[ib]); t_ba=sinkhorn_transport(bfm[ib],af[ia])
    nn_a=torch.cdist(af,af[ia]).argmin(1); nn_b=torch.cdist(bfm,bfm[ib]).argmin(1)
    interp=(0.5*(((1-eta)*af+eta*t_ab[nn_a])+(eta*bfm+(1-eta)*t_ba[nn_b]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    sm={s:(interp if s==L else F.interpolate(interp,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
    return decode_with_style_maps(d["blob_a"], sm)
def interp_lin_pixel(d, eta):
    sm_a=build_style_maps(d["bf_a"],d["si_a"],FMS); sm_b=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sm_a[s]+eta*sm_b[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])

FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}
METHODS = list(FNS)
random.seed(1)
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print("\nsetup done on", device)
os.chdir('/content'); os.system('rm -rf texture-interpolation')
os.system('git clone -q --depth 1 https://github.com/JonathanVacher/texture-interpolation.git')
VACHER_DIR = '/content/texture-interpolation/texture-synthesis-algorithm'
os.chdir(CNT_DIR)

TMPV = '/content/vacher_tmp'; os.makedirs(TMPV, exist_ok=True)
def prep_square(path, out, size=IMG_SIZE):
    Image.open(path).convert('RGB').resize((size,size), Image.BILINEAR).save(out); return out

def vacher_interp(pathA, pathB, weight=0.5, model_name='tex_wasser',
                  n_iter=30, bfgs_iter=20, shape='256,256', seed=0):
    cwd = os.getcwd(); os.chdir(VACHER_DIR)
    if VACHER_DIR not in sys.path: sys.path.insert(0, VACHER_DIR)
    for mod in [m for m in list(sys.modules) if m.startswith(('models','options'))]:
        del sys.modules[mod]
    from options.base_options import BaseOptions
    from models import create_model, utils
    argv_bak = sys.argv
    sys.argv = ['run_synthesis.py','--input_tex_1',str(pathA),'--input_tex_2',str(pathB),
                '--weight_1',str(weight),'--model',model_name,'--n_iter',str(n_iter),
                '--bfgs_iter',str(bfgs_iter),'--shape',shape,'--seed',str(seed),
                '--display_freq','100000']
    try:
        opt = BaseOptions().parse()
        m = create_model(opt); m.get_net_and_loss()
        for _ in range(opt.n_iter+1): m.optimize_parameters()
        arr = utils.tensor2im(m.output_tex.data.clamp_(0,1))
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB').resize((IMG_SIZE,)*2)
    finally:
        sys.argv = argv_bak; os.chdir(cwd)

OUT = f'{BASE}/compute_cost'; os.makedirs(OUT, exist_ok=True)
print("ready")

In [ ]:
import gc

def timed(fn, *a, warmup=1, reps=5, **kw):
    """Median wall time and peak GPU memory, after warmup, with sync."""
    for _ in range(warmup): fn(*a, **kw)
    torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    ts=[]
    for _ in range(reps):
        torch.cuda.synchronize(); t0=time.perf_counter()
        out = fn(*a, **kw)
        torch.cuda.synchronize(); ts.append(time.perf_counter()-t0)
    return float(np.median(ts)), torch.cuda.max_memory_allocated()/2**20, out

N_TIME = 20
rngt = random.Random(555)
pairs_t = [(dtd_path(*rngt.sample(cats,1)*1, s=60000+i),
            dtd_path(*rngt.sample(cats,1)*1, s=61000+i)) for i in range(N_TIME)]
pairs_t = []
for i in range(N_TIME):
    a,b = rngt.sample(cats,2)
    pairs_t.append((dtd_path(a,60000+i), dtd_path(b,61000+i)))

rows=[]
for k,(pa,pb) in enumerate(pairs_t):
    qa,qb = prep_square(pa,f'{TMPV}/A.png'), prep_square(pb,f'{TMPV}/B.png')

    # --- ours: split encode from interpolate+decode ---
    t_enc,_,_ = timed(lambda: build_d(cnt_load(qa), cnt_load(qb)))
    d = build_d(cnt_load(qa), cnt_load(qb))
    rec = dict(pair=k, encode_s=t_enc)
    for m,fn in FNS.items():
        t,mem,_ = timed(lambda: fn(d,0.5))
        rec[f'{m}_interp_s'] = t
        rec[f'{m}_total_s']  = t_enc + t
        rec[f'{m}_mem_MB']   = mem

    # --- Vacher: single call, nothing amortisable ---
    for vm in ['tex_wasser','tex_gram','tex_gauss']:
        t,mem,_ = timed(lambda: vacher_interp(qa,qb,0.5,vm,n_iter=30), warmup=0, reps=1)
        rec[f'{vm}_interp_s'] = t; rec[f'{vm}_total_s'] = t; rec[f'{vm}_mem_MB'] = mem
    rows.append(rec)
    print(f"  {k+1}/{N_TIME}")

dft = pd.DataFrame(rows); dft.to_csv(f'{OUT}/timing_n{N_TIME}.csv', index=False)
ALL_M = list(FNS) + ['tex_wasser','tex_gram','tex_gauss']

print(f"\n=== COST PER MIDPOINT (n={N_TIME}, A100) ===")
print(f"  encode both sources: {dft.encode_s.median():.3f}s  (amortised over all eta for our methods)")
print(f"\n  {'method':16} {'interp s':>10} {'total s':>9} {'peak MB':>9} {'vs ot_sym':>11}")
base = dft['ot_sym_total_s'].median()
for m in ALL_M:
    tot = dft[f'{m}_total_s'].median()
    print(f"  {m:16} {dft[f'{m}_interp_s'].median():10.3f} {tot:9.3f} "
          f"{dft[f'{m}_mem_MB'].median():9.0f} {tot/base:10.1f}x")

In [ ]:
# a path needs many eta values; our methods encode once, Vacher re-optimises per point
K_PATH = 11
print(f"=== COST OF A {K_PATH}-POINT PATH ===")
print(f"  {'method':16} {'per path s':>12} {'per hour':>10}")
for m in ALL_M:
    if m in FNS:
        t = dft.encode_s.median() + K_PATH*dft[f'{m}_interp_s'].median()
    else:
        t = K_PATH*dft[f'{m}_interp_s'].median()
    print(f"  {m:16} {t:12.2f} {3600/t:10.0f}")

# resolution scaling
print(f"\n=== SCALING WITH RESOLUTION (single midpoint) ===")
pa,pb = pairs_t[0]
print(f"  {'size':>6} " + "".join(f"{m[:11]:>13}" for m in ['ot','ot_sym','tex_wasser']))
for size in [128, 256, 512]:
    qa = prep_square(pa, f'{TMPV}/A{size}.png', size)
    qb = prep_square(pb, f'{TMPV}/B{size}.png', size)
    line = f"  {size:6d} "
    for m in ['ot','ot_sym','tex_wasser']:
        try:
            if m in FNS:
                if size != IMG_SIZE:
                    line += f"{'n/a':>13}"; continue
                d = build_d(cnt_load(qa), cnt_load(qb))
                t,_,_ = timed(lambda: FNS[m](d,0.5))
            else:
                t,_,_ = timed(lambda: vacher_interp(qa,qb,0.5,m,n_iter=30,
                                                    shape=f'{size},{size}'), warmup=0, reps=1)
            line += f"{t:13.2f}"
        except Exception as e:
            line += f"{'fail':>13}"
    print(line)

In [ ]:
import timm
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255.
            for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device)
        b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
def dino_dist(p1,p2):
    f = dino_feats([p1,p2]); return float(np.linalg.norm(f[0]-f[1]))

N_ORD = 20
rngo = random.Random(808)
pairs_o = []
for i in range(N_ORD):
    a,b = rngo.sample(cats,2)
    pairs_o.append((dtd_path(a,70000+i), dtd_path(b,71000+i)))

rows=[]; t0=time.time()
for k,(pa,pb) in enumerate(pairs_o):
    qa = prep_square(pa, f'{TMPV}/oA.png'); qb = prep_square(pb, f'{TMPV}/oB.png')
    A, B = Image.open(qa), Image.open(qb)
    rec = dict(pair=k, ab_dist=dino_dist(A,B))

    # ours
    d_ab = build_d(cnt_load(qa), cnt_load(qb))
    d_ba = build_d(cnt_load(qb), cnt_load(qa))
    for m,fn in FNS.items():
        m_ab, m_ba = to_pil(fn(d_ab,0.5)), to_pil(fn(d_ba,0.5))
        rec[f'{m}_swap'] = dino_dist(m_ab, m_ba)
        rec[f'{m}_seed'] = dino_dist(to_pil(fn(d_ab,0.5)), m_ab)   # stochastic floor

    # Vacher: same weight, swapped argument order; plus a seed-only control
    for vm in ['tex_wasser','tex_gram','tex_gauss']:
        v_ab  = vacher_interp(qa, qb, 0.5, vm, n_iter=30, seed=0)
        v_ba  = vacher_interp(qb, qa, 0.5, vm, n_iter=30, seed=0)
        v_ab2 = vacher_interp(qa, qb, 0.5, vm, n_iter=30, seed=1)
        rec[f'{vm}_swap'] = dino_dist(v_ab, v_ba)
        rec[f'{vm}_seed'] = dino_dist(v_ab, v_ab2)
        if k == 0:
            v_ab.save(f'{OUT}/order_{vm}_AB.png'); v_ba.save(f'{OUT}/order_{vm}_BA.png')
    rows.append(rec)
    print(f"  {k+1}/{N_ORD}  ({(time.time()-t0)/60:.1f}m)")

dfo = pd.DataFrame(rows); dfo.to_csv(f'{OUT}/order_dependence_n{N_ORD}.csv', index=False)

ALLM = list(FNS) + ['tex_wasser','tex_gram','tex_gauss']
print(f"\n=== ORDER DEPENDENCE (n={N_ORD}), DINOv2 distance / |A-B| ===")
print(f"  |A-B| median = {dfo.ab_dist.median():.1f}\n")
print(f"  {'method':16} {'swap A/B':>10} {'seed only':>11} {'swap/seed':>11}")
for m in ALLM:
    sw = (dfo[f'{m}_swap']/dfo.ab_dist).median()
    sd = (dfo[f'{m}_seed']/dfo.ab_dist).median()
    ratio = sw/sd if sd > 1e-6 else float('inf')
    print(f"  {m:16} {sw:10.3f} {sd:11.3f} {ratio:11.2f}")
print("\n  swap/seed near 1 = the swap is indistinguishable from run-to-run noise (symmetric)")
print("  swap/seed >> 1    = argument order genuinely changes the output")

In [ ]:
import matplotlib.pyplot as plt
pa, pb = pairs_o[0]
qa, qb = prep_square(pa, f'{TMPV}/oA.png'), prep_square(pb, f'{TMPV}/oB.png')
A, B = Image.open(qa), Image.open(qb)
d_ab = build_d(cnt_load(qa), cnt_load(qb)); d_ba = build_d(cnt_load(qb), cnt_load(qa))

ROWS = [('OT',     to_pil(interp_ot(d_ab,0.5)),     to_pil(interp_ot(d_ba,0.5))),
        ('OT-sym', to_pil(interp_ot_sym(d_ab,0.5)), to_pil(interp_ot_sym(d_ba,0.5)))]
for vm,lbl in [('tex_wasser','Wasserstein'),('tex_gram','Gram'),('tex_gauss','Gaussian')]:
    ROWS.append((lbl, Image.open(f'{OUT}/order_{vm}_AB.png'), Image.open(f'{OUT}/order_{vm}_BA.png')))

fig, ax = plt.subplots(len(ROWS), 4, figsize=(13, 3.2*len(ROWS)))
for r,(nm,ab,ba) in enumerate(ROWS):
    for c,(lbl,im) in enumerate([('A',A), ('M(A,B)',ab), ('M(B,A)',ba), ('B',B)]):
        ax[r,c].imshow(im); ax[r,c].axis('off')
        if r==0: ax[r,c].set_title(lbl, fontsize=11)
    ax[r,0].text(-0.12,0.5,nm, transform=ax[r,0].transAxes, rotation=90,
                 va='center', ha='center', fontsize=11, weight='bold')
plt.suptitle("Argument order changes the midpoint for every method except OT-sym", fontsize=13)
plt.tight_layout(); plt.savefig(f'{OUT}/order_dependence_figure.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
!pip install -q plenoptic einops
import os, sys

# Inception (realism observer #2)
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255.
            for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device)
        b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)

# Portilla-Simoncelli
import plenoptic as po
ps_model = po.models.PortillaSimoncelli((IMG_SIZE,IMG_SIZE), n_scales=4,
                                        n_orientations=4, spatial_corr_width=7).to(device)
def _raw_ps(pil):
    g = pil.convert('L').resize((IMG_SIZE,)*2)
    x = torch.from_numpy(np.asarray(g).copy()).float()[None,None].to(device)/255.
    with torch.no_grad(): return ps_model(x).detach().cpu().numpy().ravel()
VALID = ~np.isnan(_raw_ps(Image.effect_noise((256,256),60)))
def ps_stats(pil): return _raw_ps(pil)[VALID]

# VORTEX — evict Vacher's 'models' package first
os.system('git clone -q https://github.com/scabini/VORTEX.git /content/VORTEX 2>/dev/null')
for m in [m for m in list(sys.modules) if m=='models' or m.startswith('models.')]:
    del sys.modules[m]
sys.path = [p for p in sys.path if 'texture-synthesis-algorithm' not in p]
cwd = os.getcwd(); os.chdir('/content/VORTEX'); sys.path.insert(0,'/content/VORTEX')
from models import VORTEX
from torchvision import transforms
vortex_model = VORTEX('beitv2_base_patch16_224.in1k_ft_in22k_in1k', 224)
vortex_model.to(device); vortex_model.eval()
for p in vortex_model.parameters(): p.requires_grad_(False)
vtf = transforms.Compose([transforms.ToTensor(),
      transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
      transforms.Resize((224,224))])
def vortex_feats(pil):
    with torch.no_grad():
        return vortex_model(vtf(pil.convert('RGB')).unsqueeze(0).to(device)).cpu().numpy().ravel()
os.chdir(cwd)

a = Image.effect_noise((256,256),60).convert('RGB'); b = Image.effect_noise((256,256),20).convert('RGB')
print(f"PS     self {np.linalg.norm(ps_stats(a)-ps_stats(a)):.1e}  A-B {np.linalg.norm(ps_stats(a)-ps_stats(b)):.3f}")
print(f"VORTEX self {np.linalg.norm(vortex_feats(a)-vortex_feats(a)):.1e}  A-B {np.linalg.norm(vortex_feats(a)-vortex_feats(b)):.3f}")
print(f"PS stats: {VALID.sum()} valid")

In [ ]:
OUT = f'{BASE}/observer_consensus'; os.makedirs(OUT, exist_ok=True)
N_OBS = 100
rngo2 = random.Random(4242)
pairs_o2 = []
while len(pairs_o2) < N_OBS:
    a_,b_ = rngo2.sample(cats,2)
    pairs_o2.append((dtd_path(a_,20000+len(pairs_o2)), dtd_path(b_,40000+len(pairs_o2))))

# generate once
mids = {m: [] for m in METHODS}; srcs = []
t0=time.time()
for k,(pa,pb) in enumerate(pairs_o2):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d = build_d(cnt_load(A), cnt_load(B))
    for m,fn in FNS.items(): mids[m].append(to_pil(fn(d,0.5)))
    srcs.append((A,B))
    if k%25==0: print(f"  {k}/{N_OBS} ({(time.time()-t0)/60:.1f}m)")

def manifold(featfn, K=3, n=400):
    imgs=[Image.open(dtd_path(random.choice(cats),50000+i)).convert('RGB').resize((IMG_SIZE,)*2) for i in range(n)]
    Phi=featfn(imgs); Phi=Phi[~np.isnan(Phi).any(axis=1)]
    D=np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad=np.sort(D,axis=1)[:,K-1]; keep=rad<=np.median(rad)
    Pr,rr = Phi[keep], rad[keep]
    return lambda f: float(np.max(rr/(np.linalg.norm(Pr-f,axis=1)+1e-9)))

r_inc, r_dino = manifold(inc_feats), manifold(dino_feats)

def fid_pair(featfn, pil, A, B):
    """source-fidelity: mean distance to the two sources, scaled by their separation"""
    f, fA, fB = featfn([pil])[0], featfn([A])[0], featfn([B])[0]
    dA, dB = np.linalg.norm(f-fA), np.linalg.norm(f-fB)
    return 0.5*(dA+dB)/(np.linalg.norm(fA-fB)+1e-9)
def fid_single(fn, pil, A, B):
    f, fA, fB = fn(pil), fn(A), fn(B)
    dA, dB = np.linalg.norm(f-fA), np.linalg.norm(f-fB)
    return 0.5*(dA+dB)/(np.linalg.norm(fA-fB)+1e-9)

rows=[]
for k in range(len(pairs_o2)):
    A,B = srcs[k]
    rec = dict(pair=k)
    fI = inc_feats([mids[m][k] for m in METHODS])
    fD = dino_feats([mids[m][k] for m in METHODS])
    for i,m in enumerate(METHODS):
        rec[f'realism_inc__{m}']  =  r_inc(fI[i])                        # higher better
        rec[f'realism_dino__{m}'] =  r_dino(fD[i])
        rec[f'fidelity_ps__{m}']  = -fid_single(ps_stats, mids[m][k], A, B)   # negate: higher better
        rec[f'fidelity_vx__{m}']  = -fid_single(vortex_feats, mids[m][k], A, B)
        rec[f'fidelity_dino__{m}']= -fid_pair(dino_feats, mids[m][k], A, B)
    rows.append(rec)
    if k%25==0: print(f"  scoring {k}/{len(pairs_o2)}")

dfo = pd.DataFrame(rows); dfo.to_csv(f'{OUT}/observer_consensus_n{len(dfo)}.csv', index=False)
print(f"\nsaved n={len(dfo)}")

In [ ]:
from scipy.stats import kendalltau
OBS = ['realism_inc','realism_dino','fidelity_ps','fidelity_vx','fidelity_dino']

print(f"=== METHOD RANKING PER OBSERVER (best first), n={len(dfo)} ===")
rank = {}
for o in OBS:
    order = sorted(METHODS, key=lambda m: -dfo[f'{o}__{m}'].mean())
    rank[o] = {m:i for i,m in enumerate(order)}
    print(f"  {o:16}: " + "  >  ".join(order))

print(f"\n=== OBSERVER AGREEMENT (Kendall tau on ranks) ===")
print(f"  {'':16} " + "".join(f"{o[:13]:>15}" for o in OBS))
for o1 in OBS:
    line = f"  {o1:16} "
    for o2 in OBS:
        t = kendalltau([rank[o1][m] for m in METHODS], [rank[o2][m] for m in METHODS]).correlation
        line += f"{t:15.2f}"
    print(line)

print(f"\n=== ot_sym win-rate vs each baseline, per observer ===")
print(f"  {'observer':16} " + "".join(f"{b[:13]:>15}" for b in ['ot','pixel_linear','texton_linear']))
for o in OBS:
    line = f"  {o:16} "
    for base in ['ot','pixel_linear','texton_linear']:
        a,b = dfo[f'{o}__ot_sym'], dfo[f'{o}__{base}']
        line += f"{100*float((a>b).mean()):14.0f}%"
    print(line)

print(f"\n=== LEAVE-ONE-OBSERVER-OUT ===")
for drop in OBS:
    kept = [o for o in OBS if o!=drop]
    res = []
    for base in ['pixel_linear','texton_linear']:
        ok = all(dfo[f'{o}__ot_sym'].mean() > dfo[f'{o}__{base}'].mean() for o in kept)
        res.append(f"{base}:{'Y' if ok else 'N'}")
    print(f"  drop {drop:16} -> ot_sym ahead of  " + "   ".join(res))

print(f"\n=== PARETO: does any method dominate another on all five? ===")
import itertools
mu = {m: np.array([dfo[f'{o}__{m}'].mean() for o in OBS]) for m in METHODS}
found=False
for m1,m2 in itertools.permutations(METHODS,2):
    if np.all(mu[m1]>=mu[m2]) and np.any(mu[m1]>mu[m2]):
        print(f"  {m1} dominates {m2}"); found=True
if not found: print("  no method dominates any other — all four are on the Pareto front")

In [ ]:
FIG = f'{BASE}/slide_figs'; os.makedirs(FIG, exist_ok=True)

# pick a dotted → honeycombed pair; try a few and keep the one where OT diverges most
cands = [(dtd_path('dotted', 700+i), dtd_path('honeycombed', 800+i)) for i in range(6)]
best = None
for i,(pa,pb) in enumerate(cands):
    A = Image.open(pa).convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(pb).convert('RGB').resize((IMG_SIZE,)*2)
    d_ab = build_d(cnt_load(A), cnt_load(B))
    d_ba = build_d(cnt_load(B), cnt_load(A))
    ot_ab, ot_ba = to_pil(interp_ot(d_ab,0.5)), to_pil(interp_ot(d_ba,0.5))
    sy_ab, sy_ba = to_pil(interp_ot_sym(d_ab,0.5)), to_pil(interp_ot_sym(d_ba,0.5))
    f = dino_feats([ot_ab, ot_ba, sy_ab, sy_ba, A, B])
    ab = np.linalg.norm(f[4]-f[5]) + 1e-9
    d_ot  = np.linalg.norm(f[0]-f[1])/ab      # how much OT moves under a swap
    d_sym = np.linalg.norm(f[2]-f[3])/ab      # how much OT-sym moves
    print(f"  pair {i}: OT swap {d_ot:.3f}   OT-sym swap {d_sym:.3f}   gap {d_ot-d_sym:+.3f}")
    if best is None or (d_ot - d_sym) > best[0]:
        best = (d_ot-d_sym, i, pa, pb, A, B, ot_ab, ot_ba, sy_ab, sy_ba, d_ot, d_sym)

gap, idx, pa, pb, A, B, ot_ab, ot_ba, sy_ab, sy_ba, d_ot, d_sym = best
print(f"\nusing pair {idx}:  OT swap {d_ot:.3f}  vs  OT-sym swap {d_sym:.3f}")

# --- the four panels for slide 10 ---
for name, im in [('01_ot_AB',ot_ab), ('02_ot_BA',ot_ba), ('03_otsym_AB',sy_ab), ('04_otsym_BA',sy_ba)]:
    im.resize((512,512), Image.LANCZOS).save(f'{FIG}/slide10_{name}.png')
# --- and the two sources, in case you want them on the slide ---
A.resize((512,512), Image.LANCZOS).save(f'{FIG}/slide10_00_sourceA.png')
B.resize((512,512), Image.LANCZOS).save(f'{FIG}/slide10_05_sourceB.png')

# --- a combined strip, as a fallback if you'd rather paste one image ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 6, figsize=(19, 3.4))
panels = [('A', A), ('OT  M(A,B)', ot_ab), ('OT  M(B,A)', ot_ba),
          ('OT-sym  M(A,B)', sy_ab), ('OT-sym  M(B,A)', sy_ba), ('B', B)]
for a,(lbl,im) in zip(ax, panels):
    a.imshow(im); a.axis('off')
    a.set_title(lbl, fontsize=12, color=('#B85042' if 'sym' in lbl else 'black'))
plt.tight_layout(); plt.savefig(f'{FIG}/slide10_strip.png', dpi=150, bbox_inches='tight'); plt.show()

print(f"\nsaved to {FIG}")
print("  slide10_01_ot_AB.png … slide10_04_otsym_BA.png  → the four dashed boxes")
print("  slide10_strip.png                                → single-image alternative")

In [ ]:
OUT = f'{BASE}/texture_search'; os.makedirs(OUT, exist_ok=True)
import itertools, pickle

N_DICT = 40
ETAS_IDX = [0.0, 0.25, 0.5, 0.75, 1.0]
rng_s = random.Random(31)
dict_paths = [dtd_path(rng_s.choice(cats), 80000+i) for i in range(N_DICT)]

def style_summary(sm_flat):
    """mean+std per channel of a style map, flattened -> the index key."""
    mu, sd = sm_flat.mean(0), sm_flat.std(0)
    v = torch.cat([mu, sd]).detach().cpu().numpy()
    return v / (np.linalg.norm(v) + 1e-9)

def encode_cached(path):
    blob, bf, si = get_gen_inputs(cnt_load(path))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    C = sm.shape[0]
    return dict(blob=blob, bf=bf, si=si, flat=sm.permute(1,2,0).reshape(-1,C))

print(f"encoding {N_DICT} dictionary textures...")
t0=time.time()
ENC = [encode_cached(p) for p in dict_paths]
print(f"  {time.time()-t0:.1f}s")

pairs_idx = list(itertools.combinations(range(N_DICT), 2))
print(f"{len(pairs_idx)} pairs x {len(ETAS_IDX)} eta = {len(pairs_idx)*len(ETAS_IDX)} index entries")

index_keys, index_meta = [], []
t0=time.time()
for n,(i,j) in enumerate(pairs_idx):
    af, bf_ = ENC[i]['flat'], ENC[j]['flat']
    N = af.shape[0]; ns = min(4096, N)
    ia = torch.randperm(N, device=device)[:ns]; ib = torch.randperm(N, device=device)[:ns]
    with torch.no_grad():                                  # transport computed ONCE per pair
        t_ab = sinkhorn_transport(af[ia], bf_[ib])
        t_ba = sinkhorn_transport(bf_[ib], af[ia])
    nn_a = torch.cdist(af, af[ia]).argmin(1)
    nn_b = torch.cdist(bf_, bf_[ib]).argmin(1)
    Ta, Tb = t_ab[nn_a], t_ba[nn_b]
    for e in ETAS_IDX:                                     # every eta is then a lerp
        interp = 0.5*(((1-e)*af + e*Ta) + (e*bf_ + (1-e)*Tb))
        index_keys.append(style_summary(interp))
        index_meta.append((i, j, e))
    if n % 100 == 0:
        el=time.time()-t0
        print(f"  {n}/{len(pairs_idx)}  ({el/60:.1f}m, eta {el/(n+1)*(len(pairs_idx)-n-1)/60:.0f}m)")

INDEX = np.stack(index_keys)
pickle.dump({'keys':INDEX,'meta':index_meta,'paths':dict_paths,'etas':ETAS_IDX},
            open(f'{OUT}/index_{N_DICT}.pkl','wb'))
print(f"\nindex: {INDEX.shape}   built in {(time.time()-t0)/60:.1f} min")
print(f"the same index with Vacher's Wasserstein: "
      f"{len(pairs_idx)*len(ETAS_IDX)*53.1/3600:.0f} hours")

In [ ]:
def summarise_texture(path_or_pil):
    _, bf, si = get_gen_inputs(cnt_load(path_or_pil))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    C = sm.shape[0]
    return style_summary(sm.permute(1,2,0).reshape(-1,C))

def search(qvec, topk=5):
    d = 1.0 - INDEX @ qvec
    o = np.argsort(d)[:topk]
    return [(index_meta[k], float(d[k])) for k in o]

N_SYN = 30
rng_t = random.Random(77)
rows=[]
for t in range(N_SYN):
    i, j = rng_t.sample(range(N_DICT), 2)
    e_true = rng_t.choice([0.25, 0.5, 0.75])
    d = build_d(cnt_load(dict_paths[i]), cnt_load(dict_paths[j]))
    target = to_pil(interp_ot_sym(d, e_true))          # a genuine midpoint, decoded
    hits = search(summarise_texture(target), topk=5)
    (pi, pj, pe), dist = hits[0]
    exact = ({pi,pj} == {i,j})
    top5  = any({a,b} == {i,j} for (a,b,_),_ in hits)
    rows.append(dict(t=t, true_i=i, true_j=j, true_eta=e_true,
                     got_i=pi, got_j=pj, got_eta=pe,
                     pair_top1=exact, pair_top5=top5,
                     eta_err=abs(pe-e_true) if exact else np.nan, dist=dist))
    if t % 10 == 0: print(f"  {t}/{N_SYN}")

dfs_ = pd.DataFrame(rows)
print(f"\n=== RECOVERY ON SYNTHETIC TARGETS (n={N_SYN}) ===")
print(f"  correct pair, top-1 : {100*dfs_.pair_top1.mean():.0f}%")
print(f"  correct pair, top-5 : {100*dfs_.pair_top5.mean():.0f}%")
print(f"  |eta error| when the pair is right : {dfs_.eta_err.median():.3f}")
print(f"  chance (top-1)      : {100/len(pairs_idx):.2f}%")

In [ ]:
i, j, e_true = 0, 1, 0.5
af, bf_ = ENC[i]['flat'], ENC[j]['flat']
N = af.shape[0]; ns = min(4096, N)
ia = torch.randperm(N, device=device)[:ns]; ib = torch.randperm(N, device=device)[:ns]
with torch.no_grad():
    t_ab = sinkhorn_transport(af[ia], bf_[ib]); t_ba = sinkhorn_transport(bf_[ib], af[ia])
Ta = t_ab[torch.cdist(af, af[ia]).argmin(1)]; Tb = t_ba[torch.cdist(bf_, bf_[ib]).argmin(1)]
interp = 0.5*(((1-e_true)*af + e_true*Ta) + (e_true*bf_ + (1-e_true)*Tb))

k_pre  = style_summary(interp)                                  # the index key
d = build_d(cnt_load(dict_paths[i]), cnt_load(dict_paths[j]))
k_post = summarise_texture(to_pil(interp_ot_sym(d, e_true)))    # the query

print(f"cosine distance, same midpoint pre- vs post-decode : {1 - float(k_pre @ k_post):.4f}")
print(f"distance from that key to its nearest index entry  : {float(np.sort(1 - INDEX @ k_pre)[1]):.4f}")
print(f"distance from the query to its nearest index entry : {float(np.sort(1 - INDEX @ k_post)[0]):.4f}")

In [ ]:
OUT = f'{BASE}/texture_search'; os.makedirs(OUT, exist_ok=True)
import itertools, pickle

N_DICT = 40
ETAS_IDX = [0.0, 0.25, 0.5, 0.75, 1.0]
rng_s = random.Random(31)
dict_paths = [dtd_path(rng_s.choice(cats), 80000+i) for i in range(N_DICT)]

# ---------- descriptor: per-channel quantiles + channel covariance ----------
QS = torch.tensor([0.1, 0.25, 0.5, 0.75, 0.9], device=device)
def style_descriptor(flat, n_proj=64, seed=0):
    """flat: (P, C) style-map pixels. Richer than mean+std:
       quantiles capture the marginal shape, a fixed random projection of the
       channel covariance captures how channels co-vary."""
    g = torch.Generator(device=device).manual_seed(seed)
    C = flat.shape[1]
    q = torch.quantile(flat, QS, dim=0).reshape(-1)                 # 5C
    x = flat - flat.mean(0, keepdim=True)
    W = torch.randn(C, n_proj, generator=g, device=device) / np.sqrt(C)
    y = x @ W                                                        # (P, n_proj)
    cov = (y.T @ y) / max(1, y.shape[0]-1)
    iu = torch.triu_indices(n_proj, n_proj, device=device)
    v = torch.cat([q, cov[iu[0], iu[1]]])
    v = v.detach().cpu().numpy()
    return v / (np.linalg.norm(v) + 1e-9)

def encode_cached(p):
    blob, bf, si = get_gen_inputs(cnt_load(p))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    C = sm.shape[0]
    return dict(blob=blob, bf=bf, si=si, flat=sm.permute(1,2,0).reshape(-1,C))

print(f"encoding {N_DICT} dictionary textures...")
ENC = [encode_cached(p) for p in dict_paths]

# sanity: does the descriptor separate two different textures at all?
d01 = 1 - float(style_descriptor(ENC[0]['flat']) @ style_descriptor(ENC[1]['flat']))
d00 = 1 - float(style_descriptor(ENC[0]['flat']) @ style_descriptor(ENC[0]['flat']))
print(f"  descriptor check — self {d00:.5f}   two different textures {d01:.5f}")
if d01 < 0.01:
    print("  WARNING: still not discriminative; stop here rather than building the index")

pairs_idx = list(itertools.combinations(range(N_DICT), 2))
print(f"\n{len(pairs_idx)} pairs x {len(ETAS_IDX)} eta = {len(pairs_idx)*len(ETAS_IDX)} entries")

def transported(i, j, n_sub=4096):
    """Sinkhorn once per pair; every eta is then a lerp."""
    af, bf_ = ENC[i]['flat'], ENC[j]['flat']
    N = af.shape[0]; ns = min(n_sub, N)
    ia = torch.randperm(N, device=device)[:ns]; ib = torch.randperm(N, device=device)[:ns]
    with torch.no_grad():
        t_ab = sinkhorn_transport(af[ia], bf_[ib])
        t_ba = sinkhorn_transport(bf_[ib], af[ia])
    Ta = t_ab[torch.cdist(af, af[ia]).argmin(1)]
    Tb = t_ba[torch.cdist(bf_, bf_[ib]).argmin(1)]
    return af, bf_, Ta, Tb
def blend(af, bf_, Ta, Tb, e):
    return 0.5*(((1-e)*af + e*Ta) + (e*bf_ + (1-e)*Tb))

index_keys, index_meta = [], []
t0 = time.time()
for n,(i,j) in enumerate(pairs_idx):
    af, bf_, Ta, Tb = transported(i, j)
    for e in ETAS_IDX:
        index_keys.append(style_descriptor(blend(af, bf_, Ta, Tb, e)))
        index_meta.append((i, j, e))
    if n % 100 == 0:
        el = time.time()-t0
        print(f"  {n}/{len(pairs_idx)}  ({el/60:.1f}m, eta {el/(n+1)*(len(pairs_idx)-n-1)/60:.0f}m)")

INDEX = np.stack(index_keys)
pickle.dump({'keys':INDEX,'meta':index_meta,'paths':dict_paths,'etas':ETAS_IDX},
            open(f'{OUT}/index_{N_DICT}.pkl','wb'))

D = 1 - INDEX @ INDEX.T
np.fill_diagonal(D, np.inf)
print(f"\nindex {INDEX.shape}, built in {(time.time()-t0)/60:.1f} min")
print(f"  spread of entries: nearest-neighbour distance median {np.median(D.min(1)):.5f}, "
      f"overall median {np.median(D[np.triu_indices(len(D),1)]):.5f}")
print(f"  same index with Vacher's Wasserstein: {len(pairs_idx)*len(ETAS_IDX)*53.1/3600:.0f} hours")

# ---------- recovery test: targets whose answer is known ----------
def search(q, topk=5):
    d = 1 - INDEX @ q
    o = np.argsort(d)[:topk]
    return [(index_meta[k], float(d[k])) for k in o]

N_SYN = 30
rng_t = random.Random(77)
rows = []
for t in range(N_SYN):
    i, j = rng_t.sample(range(N_DICT), 2)
    e_true = rng_t.choice([0.25, 0.5, 0.75])
    af, bf_, Ta, Tb = transported(i, j)            # fresh subsample -> not the index entry
    q = style_descriptor(blend(af, bf_, Ta, Tb, e_true))
    hits = search(q, topk=5)
    (pi, pj, pe), dist = hits[0]
    exact = ({pi, pj} == {i, j})
    rows.append(dict(t=t, true_i=i, true_j=j, true_eta=e_true, got_i=pi, got_j=pj, got_eta=pe,
                     pair_top1=exact, pair_top5=any({a,b}=={i,j} for (a,b,_),_ in hits),
                     eta_err=abs(pe-e_true) if exact else np.nan, dist=dist))
    if t % 10 == 0: print(f"  recovery {t}/{N_SYN}")

dfs_ = pd.DataFrame(rows)
dfs_.to_csv(f'{OUT}/recovery_n{N_SYN}.csv', index=False)
print(f"\n=== RECOVERY ON KNOWN TARGETS (n={N_SYN}) ===")
print(f"  correct pair, top-1 : {100*dfs_.pair_top1.mean():5.0f}%")
print(f"  correct pair, top-5 : {100*dfs_.pair_top5.mean():5.0f}%")
print(f"  chance, top-1       : {100/len(pairs_idx):5.2f}%")
print(f"  |eta error| when the pair is correct : {dfs_.eta_err.median():.3f}")

In [ ]:
# 1. is the query actually different from its index entry?
i, j, e = 0, 1, 0.5
af,bf_,Ta,Tb = transported(i,j); q1 = style_descriptor(blend(af,bf_,Ta,Tb,e))
af,bf_,Ta,Tb = transported(i,j); q2 = style_descriptor(blend(af,bf_,Ta,Tb,e))
k = INDEX[index_meta.index((i,j,e))]
print(f"two independent runs of the same midpoint : {1-float(q1@q2):.7f}")
print(f"query vs its own index entry             : {1-float(q1@k):.7f}")
print(f"query vs a different pair's entry        : {1-float(q1@INDEX[index_meta.index((2,3,e))]):.7f}")

# 2. mean-centre the index so cosine is not dominated by a shared component
mu = INDEX.mean(0)
Z = INDEX - mu; Z /= (np.linalg.norm(Z,axis=1,keepdims=True)+1e-12)
def search_c(q, topk=5):
    z = q - mu; z /= (np.linalg.norm(z)+1e-12)
    o = np.argsort(1 - Z @ z)[:topk]
    return [(index_meta[k], float((1-Z@z)[k])) for k in o]
hits = search_c(q1, 5)
print("\ncentred search, top-5:", [(m, round(d,5)) for m,d in hits])
print(f"centred: nearest-neighbour spacing median "
      f"{np.median(np.sort(1-Z@Z.T,axis=1)[:,1]):.5f}")

In [ ]:
HELD = [dtd_path(rng_s.choice(cats), 95000+i) for i in range(40)]
HELD = [p for p in HELD if p not in dict_paths]

# singles baseline: each dictionary texture on its own
singles = np.stack([style_descriptor(ENC[i]['flat']) for i in range(N_DICT)])
mu_s = singles.mean(0); Zs = singles - mu_s
Zs /= (np.linalg.norm(Zs,axis=1,keepdims=True)+1e-12)

rows=[]
for n,p in enumerate(HELD):
    _, bf, si = get_gen_inputs(cnt_load(p))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    q = style_descriptor(sm.permute(1,2,0).reshape(-1, sm.shape[0]))

    zq = q - mu; zq /= (np.linalg.norm(zq)+1e-12)
    d_int = 1 - Z @ zq
    b_int = int(np.argmin(d_int)); (bi,bj,be) = index_meta[b_int]

    zs = q - mu_s; zs /= (np.linalg.norm(zs)+1e-12)
    d_sing = 1 - Zs @ zs
    b_sing = int(np.argmin(d_sing))

    rows.append(dict(path=p, best_pair=(bi,bj), best_eta=be,
                     d_interp=float(d_int[b_int]), d_single=float(d_sing[b_sing]),
                     interp_better=float(d_int[b_int]) < float(d_sing[b_sing]),
                     eta_interior=(0 < be < 1)))
    if n%10==0: print(f"  {n}/{len(HELD)}")

dh = pd.DataFrame(rows)
print(f"\n=== ARBITRARY TARGETS (n={len(dh)}) ===")
print(f"  interpolation closer than the best single texture : {100*dh.interp_better.mean():.0f}%")
print(f"  median distance   interpolation {dh.d_interp.median():.4f}   single {dh.d_single.median():.4f}")
print(f"  chosen eta is interior (not an endpoint)          : {100*dh.eta_interior.mean():.0f}%")
print(f"\n  eta distribution:"); print(dh.best_eta.value_counts().sort_index().to_string())

In [ ]:
ALL = np.vstack([INDEX, singles])
ALL_meta = index_meta + [('single', i, None) for i in range(N_DICT)]
mu_a = ALL.mean(0); Za = ALL - mu_a
Za /= (np.linalg.norm(Za,axis=1,keepdims=True)+1e-12)

wins = {'interp':0, 'single':0}
for p in HELD:
    _, bf, si = get_gen_inputs(cnt_load(p))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    q = style_descriptor(sm.permute(1,2,0).reshape(-1, sm.shape[0]))
    z = q - mu_a; z /= (np.linalg.norm(z)+1e-12)
    best = ALL_meta[int(np.argmin(1 - Za @ z))]
    wins['single' if best[0]=='single' else 'interp'] += 1
print(f"one shared index — interpolation wins {wins['interp']}/{len(HELD)}, "
      f"single wins {wins['single']}/{len(HELD)}")
print(f"  (interpolated entries are {len(INDEX)}/{len(ALL)} = {100*len(INDEX)/len(ALL):.0f}% of candidates)")

In [ ]:
OUT = f'{BASE}/texture_search'; os.makedirs(OUT, exist_ok=True)

# ---------- objective: sliced Wasserstein between two style-map clouds ----------
def sw_distance(X, Y, n_proj=128, gen=None):
    """X:(P,C) Y:(Q,C). Sliced Wasserstein-2. Differentiable in X."""
    C = X.shape[1]
    W = torch.randn(C, n_proj, device=X.device, generator=gen)
    W = W / (W.norm(dim=0, keepdim=True) + 1e-9)
    xp = (X @ W).sort(dim=0).values                       # (P, n_proj)
    yp = (Y @ W).sort(dim=0).values                       # (Q, n_proj)
    if xp.shape[0] != yp.shape[0]:                        # match sample counts
        n = min(xp.shape[0], yp.shape[0])
        idx_x = torch.linspace(0, xp.shape[0]-1, n, device=X.device).long()
        idx_y = torch.linspace(0, yp.shape[0]-1, n, device=X.device).long()
        xp, yp = xp[idx_x], yp[idx_y]
    return ((xp - yp)**2).mean()

# ---------- k-way barycentre with learnable weights ----------
def pairwise_transports(idxs, n_sub=2048):
    """For a set of dictionary indices, precompute all pairwise transported clouds.
       Returns clouds[i] (subsampled) and T[(i,j)] = where i's points go under i->j."""
    clouds, T = {}, {}
    for i in idxs:
        f = ENC[i]['flat']
        sel = torch.randperm(f.shape[0], device=device)[:n_sub]
        clouds[i] = f[sel]
    for i in idxs:
        for j in idxs:
            if i == j: continue
            with torch.no_grad():
                T[(i,j)] = sinkhorn_transport(clouds[i], clouds[j])
    return clouds, T

def barycentre(clouds, T, idxs, w):
    """Symmetric k-way displacement interpolation.
       Each member is moved toward the weighted mix of the others."""
    parts = []
    for a, i in enumerate(idxs):
        dest = torch.zeros_like(clouds[i])
        wsum = 0.0
        for b, j in enumerate(idxs):
            if i == j: continue
            dest = dest + w[b] * T[(i,j)]
            wsum = wsum + w[b]
        dest = dest / (wsum + 1e-9)
        # member i travels a fraction (1 - w_i) of the way to that destination
        parts.append(w[a] * ((w[a]) * clouds[i] + (1 - w[a]) * dest))
    return torch.stack(parts).sum(0) / (w.sum() + 1e-9)

print("objective + barycentre ready")

In [ ]:
K = 5
N_STEPS = 120
LR = 0.15

def target_cloud(path):
    _, bf, si = get_gen_inputs(cnt_load(path))
    sm = build_style_maps(bf, si, FMS)[max(FMS)][0]
    f = sm.permute(1,2,0).reshape(-1, sm.shape[0])
    sel = torch.randperm(f.shape[0], device=device)[:2048]
    return f[sel]

def shortlist(Tq, k=K):
    """cheapest first pass: the k dictionary textures closest to the target"""
    d = [float(sw_distance(ENC[i]['flat'][:2048], Tq, n_proj=32)) for i in range(N_DICT)]
    return list(np.argsort(d)[:k]), float(np.min(d))

rows = []
for n, p in enumerate(HELD):
    Tq = target_cloud(p)
    idxs, d_single = shortlist(Tq)                      # baseline 1: best single texture

    clouds, T = pairwise_transports(idxs)

    # baseline 2: best pair from these k, coarse eta grid
    best_pair = np.inf
    for a in range(K):
        for b in range(a+1, K):
            i, j = idxs[a], idxs[b]
            for e in [0.25, 0.5, 0.75]:
                mix = (1-e)*clouds[i] + e*T[(i,j)]
                best_pair = min(best_pair, float(sw_distance(mix, Tq)))

    # ours: k-way barycentre, weights optimised
    logits = torch.zeros(K, device=device, requires_grad=True)
    opt = torch.optim.Adam([logits], lr=LR)
    for step in range(N_STEPS):
        w = torch.softmax(logits, 0)
        loss = sw_distance(barycentre(clouds, T, idxs, w), Tq)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        w = torch.softmax(logits, 0)
        d_bary = float(sw_distance(barycentre(clouds, T, idxs, w), Tq))

    rows.append(dict(path=p, d_single=d_single, d_pair=best_pair, d_bary=d_bary,
                     beats_single=d_bary < d_single, beats_pair=d_bary < best_pair,
                     w_max=float(w.max()), w_entropy=float(-(w*torch.log(w+1e-9)).sum()),
                     idxs=idxs))
    if n % 5 == 0:
        print(f"  {n}/{len(HELD)}  single {d_single:.4f}  pair {best_pair:.4f}  bary {d_bary:.4f}")

db = pd.DataFrame(rows); db.to_csv(f'{OUT}/barycentre_search_n{len(db)}.csv', index=False)

print(f"\n=== k={K} BARYCENTRE SEARCH (n={len(db)}) ===")
print(f"  median SW distance   single {db.d_single.median():.4f}   "
      f"pair {db.d_pair.median():.4f}   barycentre {db.d_bary.median():.4f}")
print(f"  barycentre beats best single texture : {100*db.beats_single.mean():.0f}%")
print(f"  barycentre beats best 2-texture pair : {100*db.beats_pair.mean():.0f}%")
print(f"  weight concentration: max weight median {db.w_max.median():.2f}  "
      f"(1.0 = collapsed to one texture, {1/K:.2f} = uniform)")
print(f"  weight entropy median {db.w_entropy.median():.2f}  (max {np.log(K):.2f})")

In [ ]:
p = HELD[0]
Tq = target_cloud(p)
print(f"target cloud: shape {tuple(Tq.shape)}  mean {Tq.mean():.4f}  std {Tq.std():.4f}  "
      f"absmax {Tq.abs().max():.4f}")
d0 = ENC[0]['flat'][:2048]
print(f"dict cloud 0: mean {d0.mean():.4f}  std {d0.std():.4f}")
print(f"\nSW(target, dict0)      = {float(sw_distance(d0, Tq)):.8f}")
print(f"SW(target, target)     = {float(sw_distance(Tq, Tq)):.8f}")
print(f"SW(dict0, dict1)       = {float(sw_distance(d0, ENC[1]['flat'][:2048])):.8f}")
print(f"L2 between cloud means = {float((d0.mean(0)-Tq.mean(0)).norm()):.6f}")

idxs, _ = shortlist(Tq)
clouds, T = pairwise_transports(idxs)
for wtest, name in [(torch.eye(K, device=device)[0], 'one-hot on member 0'),
                    (torch.ones(K, device=device)/K, 'uniform')]:
    b = barycentre(clouds, T, idxs, wtest)
    print(f"  {name:22} SW to target {float(sw_distance(b, Tq)):.8f}   "
          f"SW to member0 {float(sw_distance(b, clouds[idxs[0]])):.8f}")

In [ ]:
G = torch.Generator(device=device).manual_seed(0)
C_DIM = ENC[0]['flat'].shape[1]
W_FIX = torch.randn(C_DIM, 1024, device=device, generator=G)
W_FIX = W_FIX / (W_FIX.norm(dim=0, keepdim=True) + 1e-9)

def sw_distance(X, Y, W=W_FIX):
    xp = (X @ W).sort(dim=0).values
    yp = (Y @ W).sort(dim=0).values
    if xp.shape[0] != yp.shape[0]:
        n = min(xp.shape[0], yp.shape[0])
        xi = torch.linspace(0, xp.shape[0]-1, n, device=X.device).long()
        yi = torch.linspace(0, yp.shape[0]-1, n, device=X.device).long()
        xp, yp = xp[xi], yp[yi]
    return ((xp - yp)**2).mean()

def shortlist(Tq, k=K):
    d = [float(sw_distance(ENC[i]['flat'][:2048], Tq)) for i in range(N_DICT)]
    return list(np.argsort(d)[:k]), float(np.min(d))

# consistency checks
Tq = target_cloud(HELD[0])
idxs, _ = shortlist(Tq)
clouds, T = pairwise_transports(idxs)
d_direct = float(sw_distance(clouds[idxs[0]], Tq))
d_onehot = float(sw_distance(barycentre(clouds, T, idxs, torch.eye(K,device=device)[0]), Tq))
print(f"SW(member0, target)            = {d_direct:.8f}")
print(f"SW(one-hot barycentre, target) = {d_onehot:.8f}")
print(f"  must agree — diff {abs(d_direct-d_onehot):.2e}")
print(f"\nrepeatability: {float(sw_distance(clouds[idxs[0]], Tq)):.8f}  "
      f"{float(sw_distance(clouds[idxs[0]], Tq)):.8f}")
print(f"dynamic range:")
print(f"  dict0 vs dict1        {float(sw_distance(ENC[0]['flat'][:2048], ENC[1]['flat'][:2048])):.8f}")
print(f"  dict0 vs target       {float(sw_distance(ENC[0]['flat'][:2048], Tq)):.8f}")
print(f"  target vs itself      {float(sw_distance(Tq, Tq)):.8f}")

In [ ]:
K = 5
N_STEPS = 150
LR = 0.15

rows = []
for n, p in enumerate(HELD):
    Tq = target_cloud(p)
    idxs, d_single = shortlist(Tq)
    clouds, T = pairwise_transports(idxs)

    best_pair = np.inf
    for a in range(K):
        for b in range(a+1, K):
            i, j = idxs[a], idxs[b]
            for e in [0.2,0.4,0.5,0.6,0.8]:
                mix = (1-e)*clouds[i] + e*T[(i,j)]
                best_pair = min(best_pair, float(sw_distance(mix, Tq)))

    logits = torch.zeros(K, device=device, requires_grad=True)
    opt = torch.optim.Adam([logits], lr=LR)
    for _ in range(N_STEPS):
        w = torch.softmax(logits, 0)
        loss = sw_distance(barycentre(clouds, T, idxs, w), Tq)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        w = torch.softmax(logits, 0)
        d_bary = float(sw_distance(barycentre(clouds, T, idxs, w), Tq))

    rows.append(dict(path=p, d_single=d_single, d_pair=best_pair, d_bary=d_bary,
                     beats_single=d_bary<d_single, beats_pair=d_bary<best_pair,
                     gain=1-d_bary/max(d_single,1e-12),
                     w_max=float(w.max()), w_entropy=float(-(w*torch.log(w+1e-9)).sum())))
    if n%5==0: print(f"  {n}/{len(HELD)}  single {d_single:.6f}  pair {best_pair:.6f}  bary {d_bary:.6f}")

db = pd.DataFrame(rows); db.to_csv(f'{OUT}/barycentre_search_fixed_n{len(db)}.csv', index=False)
print(f"\n=== k={K} BARYCENTRE, fixed objective (n={len(db)}) ===")
print(f"  median SW   single {db.d_single.median():.6f}   pair {db.d_pair.median():.6f}   "
      f"barycentre {db.d_bary.median():.6f}")
print(f"  beats best single : {100*db.beats_single.mean():3.0f}%     "
      f"median reduction {100*db.gain.median():.0f}%")
print(f"  beats best pair   : {100*db.beats_pair.mean():3.0f}%")
print(f"  weights: max {db.w_max.median():.2f}  entropy {db.w_entropy.median():.2f} "
      f"(uniform = {np.log(K):.2f}, collapsed = 0)")

In [ ]:
Tq = target_cloud(HELD[0])
idxs, d_single = shortlist(Tq)
clouds, T = pairwise_transports(idxs)

print("loss at fixed weight settings:")
for name, wv in [('uniform', torch.ones(K,device=device)/K),
                 ('one-hot member0', torch.eye(K,device=device)[0]),
                 ('half 0 / half 1', torch.tensor([.5,.5,0,0,0],device=device)),
                 ('two-thirds/third', torch.tensor([.67,.33,0,0,0],device=device))]:
    print(f"  {name:18} {float(sw_distance(barycentre(clouds,T,idxs,wv), Tq)):.8f}")

# what the optimiser actually does, traced
logits = torch.zeros(K, device=device, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.15)
for step in range(150):
    w = torch.softmax(logits,0)
    loss = sw_distance(barycentre(clouds,T,idxs,w), Tq)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 30 == 0:
        with torch.no_grad():
            print(f"  step {step:3d}  loss {float(loss):.8f}  w_max {float(torch.softmax(logits,0).max()):.3f}")

In [ ]:
def barycentre(clouds, T, idxs, w, anchor=0):
    """Weighted displacement interpolation onto a fixed anchor's support.
       Every member is expressed in the anchor's coordinates, then averaged.
       Well-defined for any w on the simplex; returns the anchor when w is
       one-hot on the anchor."""
    a = idxs[anchor]
    parts = []
    for b, j in enumerate(idxs):
        src = clouds[a] if j == a else T[(a, j)]     # where anchor's points go under a->j
        parts.append(w[b] * src)
    return torch.stack(parts).sum(0) / (w.sum() + 1e-9)

In [ ]:
K = 5
N_STEPS = 150
LR = 0.15

rows = []
for n, p in enumerate(HELD):
    Tq = target_cloud(p)
    idxs, d_single = shortlist(Tq)
    clouds, T = pairwise_transports(idxs)

    best_pair = np.inf
    for a in range(K):
        for b in range(a+1, K):
            i, j = idxs[a], idxs[b]
            for e in [0.2,0.4,0.5,0.6,0.8]:
                mix = (1-e)*clouds[i] + e*T[(i,j)]
                best_pair = min(best_pair, float(sw_distance(mix, Tq)))

    logits = torch.zeros(K, device=device, requires_grad=True)
    opt = torch.optim.Adam([logits], lr=LR)
    for _ in range(N_STEPS):
        w = torch.softmax(logits, 0)
        loss = sw_distance(barycentre(clouds, T, idxs, w), Tq)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        w = torch.softmax(logits, 0)
        d_bary = float(sw_distance(barycentre(clouds, T, idxs, w), Tq))

    rows.append(dict(path=p, d_single=d_single, d_pair=best_pair, d_bary=d_bary,
                     beats_single=d_bary<d_single, beats_pair=d_bary<best_pair,
                     gain_single=1-d_bary/max(d_single,1e-12),
                     gain_pair=1-d_bary/max(best_pair,1e-12),
                     w_max=float(w.max()), w_entropy=float(-(w*torch.log(w+1e-9)).sum()),
                     n_active=int((w>0.05).sum())))
    if n%5==0: print(f"  {n}/{len(HELD)}  single {d_single:.6f}  pair {best_pair:.6f}  bary {d_bary:.6f}")

db = pd.DataFrame(rows); db.to_csv(f'{OUT}/barycentre_anchored_n{len(db)}.csv', index=False)
print(f"\n=== k={K} ANCHORED BARYCENTRE (n={len(db)}) ===")
print(f"  median SW   single {db.d_single.median():.6f}   pair {db.d_pair.median():.6f}   "
      f"barycentre {db.d_bary.median():.6f}")
print(f"  beats best single : {100*db.beats_single.mean():3.0f}%   median reduction {100*db.gain_single.median():.0f}%")
print(f"  beats best pair   : {100*db.beats_pair.mean():3.0f}%   median reduction {100*db.gain_pair.median():.0f}%")
print(f"  weights: max {db.w_max.median():.2f}  entropy {db.w_entropy.median():.2f} "
      f"(uniform {np.log(K):.2f})   active members {db.n_active.median():.0f}/{K}")